# Feature-Name Anonymization Ablation — UNSW-NB15

Investigates whether the LLM selects features based on **semantic name reasoning**
or **genuine statistical patterns** — replicating the WUSTL-IIoT ablation design.

**Design:**
- Two conditions on identical data:
  - **Original**: real feature names (`dur`, `spkts`, `sbytes`, ...)
  - **Anonymized**: opaque labels `f0`, `f1`, ..., `f38`
- 3 independent seeds (LLM temperature=0.1).
- Binary classification: `label` column (0=Normal, 1=Attack).

**Dataset:** Raw UNSW-NB15 CSV files concatenated (175,341 + 82,332 rows).
Balanced to `SAMPLE_SIZE` = 10,000 (5,000 normal + 5,000 attack) — consistent
with the existing binary result in `results/result-10000-2-10000.txt`.

**Metrics:** ΔF1, Jaccard feature-selection overlap, semantic category distribution.

**Output:** `results/anon/unsw-nb15-anon-ablation.json`

## Cell 1 — Load Dataset

In [1]:
################################################################################
# Cell 1 — Load Dataset
#
# Concatenate training + testing CSVs into a single population.
# Binary label: 'label' column (0=Normal, 1=Attack).
# Numeric features only; drop ['label', 'attack_cat', 'id'].
# Balance to SAMPLE_SIZE (5,000 normal + 5,000 attack), stratified 80/20 split.
################################################################################

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tabulate import tabulate

dataset_name = 'unsw-nb15'
TRAIN_PATH   = os.path.expanduser(
    '~/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_training-set.csv'
)
TEST_PATH    = os.path.expanduser(
    '~/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_testing-set.csv'
)
LABEL_COL    = 'label'     # binary: 0=Normal, 1=Attack
DROP_COLS    = ['label', 'attack_cat', 'id']
SAMPLE_SIZE  = 5000        # per class (5K normal + 5K attack = 10K total)

df_tr = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
df_te = pd.read_csv(TEST_PATH,  encoding='utf-8-sig')
df_raw = pd.concat([df_tr, df_te], ignore_index=True)

feature_cols = [
    c for c in df_raw.select_dtypes(include=[np.number]).columns
    if c not in DROP_COLS
]

normal_all = df_raw[df_raw[LABEL_COL] == 0][feature_cols].reset_index(drop=True)
attack_all = df_raw[df_raw[LABEL_COL] == 1][feature_cols].reset_index(drop=True)

print(f'Population: {len(df_raw):,} rows  |  '
      f'Normal={len(normal_all):,}  Attack={len(attack_all):,}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

# Balance to SAMPLE_SIZE per class
normal_bal = normal_all.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
attack_bal = attack_all.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# 80/20 split
normal_df_train, normal_df_test = train_test_split(
    normal_bal, test_size=0.2, random_state=42)
attack_df_train, attack_df_test = train_test_split(
    attack_bal, test_size=0.2, random_state=42)

normal_df_train = normal_df_train.reset_index(drop=True)
normal_df_test  = normal_df_test.reset_index(drop=True)
attack_df_train = attack_df_train.reset_index(drop=True)
attack_df_test  = attack_df_test.reset_index(drop=True)

data = [
    ['Normal', len(normal_df_train) + len(normal_df_test),
     len(normal_df_train), len(normal_df_test)],
    ['Attack', len(attack_df_train) + len(attack_df_test),
     len(attack_df_train), len(attack_df_test)],
]
print()
print(tabulate(data, headers=['Class', 'Total', 'Train', 'Test'], tablefmt='grid'))


Population: 257,673 rows  |  Normal=93,000  Attack=164,673
Features (39): ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

+---------+---------+---------+--------+
| Class   |   Total |   Train |   Test |
+=========+=========+=========+========+
| Normal  |    5000 |    4000 |   1000 |
+---------+---------+---------+--------+
| Attack  |    5000 |    4000 |   1000 |
+---------+---------+---------+--------+


## Cell 2 — Anonymization Map + Semantic Categories

In [2]:
################################################################################
# Cell 2 — Anonymization Map and Semantic Categories
################################################################################

anon_map    = {name: f'f{i}' for i, name in enumerate(feature_cols)}
reverse_map = {f'f{i}': name for i, name in enumerate(feature_cols)}

# Semantic categories for UNSW-NB15 features
SEMANTIC_CATEGORIES = {
    'flow':       ['dur', 'rate', 'trans_depth', 'response_body_len'],
    'packet':     ['spkts', 'dpkts', 'sloss', 'dloss', 'sinpkt', 'dinpkt',
                   'sjit', 'djit'],
    'byte':       ['sbytes', 'dbytes', 'sload', 'dload', 'smean', 'dmean'],
    'tcp_state':  ['swin', 'dwin', 'stcpb', 'dtcpb', 'tcprtt', 'synack',
                   'ackdat'],
    'ttl':        ['sttl', 'dttl'],
    'connection': ['ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
                   'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
                   'ct_src_ltm', 'ct_srv_dst'],
    'protocol':   ['is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
                   'is_sm_ips_ports'],
}


def get_category(feature_name: str) -> str:
    real = reverse_map.get(feature_name, feature_name)
    for cat, members in SEMANTIC_CATEGORIES.items():
        if real in members:
            return cat
    return 'unknown'


print(f'Feature count: {len(feature_cols)}')
print('\nAnonymization map:')
for k, v in anon_map.items():
    print(f'  {k:25s} \u2192 {v}')
print('\nSemantic categories:')
for cat, members in SEMANTIC_CATEGORIES.items():
    covered = [m for m in members if m in feature_cols]
    print(f'  {cat:12s}: {covered}')


Feature count: 39

Anonymization map:
  dur                       → f0
  spkts                     → f1
  dpkts                     → f2
  sbytes                    → f3
  dbytes                    → f4
  rate                      → f5
  sttl                      → f6
  dttl                      → f7
  sload                     → f8
  dload                     → f9
  sloss                     → f10
  dloss                     → f11
  sinpkt                    → f12
  dinpkt                    → f13
  sjit                      → f14
  djit                      → f15
  swin                      → f16
  stcpb                     → f17
  dtcpb                     → f18
  dwin                      → f19
  tcprtt                    → f20
  synack                    → f21
  ackdat                    → f22
  smean                     → f23
  dmean                     → f24
  trans_depth               → f25
  response_body_len         → f26
  ct_srv_src                → f27
  ct_state_ttl      

## Cell 3 — Fixed Sample Retrieval via BGE-M3

In [3]:
################################################################################
# Cell 3 — Representative Samples via BGE-M3
#
# Subsample up to MAX_EMBED rows per class, embed, select top N_REPR by cosine
# similarity to mean. Samples are fixed here (seed=42) and reused across all
# seeds in the multi-seed loop — isolating the effect of LLM randomness.
################################################################################

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

N_RETRIEVAL = 10
MAX_EMBED   = 100

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64},
)


def get_repr_bge(df: pd.DataFrame, n: int = 10,
                 max_embed: int = 100, seed: int = 42) -> pd.DataFrame:
    sample = df.sample(n=min(max_embed, len(df)), random_state=seed)
    docs   = [str(row.tolist()) for _, row in sample.iterrows()]
    vecs   = np.array(embeddings.embed_documents(docs))
    mean_v = vecs.mean(axis=0)
    norms  = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_v)
    sims   = (vecs @ mean_v) / np.where(norms == 0, 1e-9, norms)
    top_i  = np.argsort(sims)[::-1][:n]
    return sample.iloc[top_i]


print('Computing normal representative samples...')
normal_repr = get_repr_bge(normal_df_train, N_RETRIEVAL, MAX_EMBED)
print('Computing attack representative samples...')
attack_repr = get_repr_bge(attack_df_train, N_RETRIEVAL, MAX_EMBED)

# Build entry dicts for both conditions (computed once, reused across seeds)
normal_entries_orig = {col: normal_repr[col].tolist() for col in feature_cols}
attack_entries_orig = {col: attack_repr[col].tolist() for col in feature_cols}
normal_entries_anon = {anon_map[col]: normal_repr[col].tolist() for col in feature_cols}
attack_entries_anon = {anon_map[col]: attack_repr[col].tolist() for col in feature_cols}

print(f'\u2713  {len(normal_repr)} normal + {len(attack_repr)} attack representative samples')
print(f'   Sample normal entry (first 5 features): '
      f'{ {k: normal_entries_orig[k] for k in feature_cols[:5]} }')


Computing normal representative samples...
Computing attack representative samples...
✓  10 normal + 10 attack representative samples
   Sample normal entry (first 5 features): {'dur': [0.540461, 0.584485, 0.538868, 0.897468, 1.064899, 0.539826, 0.50438, 0.562673, 0.017816, 0.020915], 'spkts': [12, 10, 10, 14, 120, 12, 14, 10, 40, 44], 'dpkts': [6, 6, 8, 10, 126, 8, 6, 6, 42, 46], 'sbytes': [4194, 534, 824, 774, 7670, 1066, 9242, 534, 2542, 2766], 'dbytes': [268, 268, 1074, 556, 14840, 762, 268, 268, 23508, 24004]}


## Cell 4 — Prompt Template

In [4]:
################################################################################
# Cell 4 — Prompt Template
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ('system',
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
""")

human_message = ('user',
"""
Analyze the following network data and generate rules for the top {k} important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
""")

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder('msgs'),
])

print('Prompt template ready.')


Prompt template ready.


## Cell 5 — Anonymization-Aware evaluate_rule Tool + LLM

In [ ]:
################################################################################
# Cell 5 — evaluate_rule Tool (anon-aware) + LLM
#
# Accepts both real names and f{i} anonymous labels via reverse_map.
# All UNSW-NB15 features are numeric; no string-comparison branch needed,
# but dtype check is kept for robustness.
################################################################################

import operator, os, dotenv
from typing import Annotated
from langchain_core.tools import tool
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from sklearn.metrics import classification_report

dotenv.load_dotenv(os.getcwd() + '/../.env')

OPERATIONS = {
    '<': operator.lt, '>': operator.gt,
    '==': operator.eq, '<=': operator.le,
    '>=': operator.ge, '!=': operator.ne,
}


@tool
def evaluate_rule(
    feature_name: Annotated[str, 'Feature name (real or anonymous f{i})'],
    value:        Annotated[str, 'Threshold value'],
    op:           Annotated[str, 'Comparison operator: ==, !=, >, <, >=, <='],
) -> float:
    """Evaluate a single threshold rule on the training set. Returns macro F1-score."""
    real_name = reverse_map.get(feature_name, feature_name)
    if op not in OPERATIONS:
        raise ValueError(f'Unsupported operator: {op}')
    op_fn = OPERATIONS[op]
    col = normal_df_train[real_name]
    if pd.api.types.is_numeric_dtype(col):
        try:
            value = float(value)
        except (ValueError, TypeError):
            pass
    else:
        value = str(value)
    datasets = {'normal': normal_df_train, 'attack': attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        col_vals = dataset[real_name].values
        for v in col_vals:
            try:
                y_pred.append('attack' if op_fn(v, value) else 'normal')
            except TypeError:
                y_pred.append('normal')
        y_true.extend([label] * len(dataset))
    report = classification_report(
        y_true, y_pred, digits=4, output_dict=True, zero_division=0
    )
    return report['macro avg']['f1-score']


model_name = 'claude-haiku-4-5-20251001'
llm = ChatAnthropic(model=model_name, temperature=0.1)
llm_with_tool = llm.bind_tools([evaluate_rule])

print(f'LLM: {model_name}, tool: evaluate_rule (anon-aware)')

## Cell 6 — run_experiment() Function

In [6]:
################################################################################
# Cell 6 — run_experiment() Function
#
# Runs N_ROUNDS of the LLM feedback loop for one condition (original/anonymized).
# Returns the final tool calls and training F1 history.
# ValidationError on missing 'value' field is caught and scored 0.0.
################################################################################

import json, time
from langchain_core.messages import HumanMessage

try:
    from openai import RateLimitError
except ImportError:
    RateLimitError = Exception

try:
    from pydantic import ValidationError as PydanticValidationError
except ImportError:
    PydanticValidationError = ValueError

N_ROUNDS           = 5
K_RULES            = 5
INTER_ROUND_SLEEP  = 5


def invoke_with_retry(chain, inputs, max_retries=6, base_delay=15.0):
    for attempt in range(max_retries):
        try:
            return chain.invoke(inputs)
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f'  \u26a0 Rate limit (attempt {attempt+1}). Sleeping {wait:.0f}s...')
            time.sleep(wait)
        except Exception:
            raise


def safe_invoke_tool(tool_call):
    """Invoke evaluate_rule; return score=0.0 on ValidationError (missing field)."""
    try:
        return evaluate_rule.invoke(tool_call)
    except (PydanticValidationError, Exception) as exc:
        if 'Field required' in str(exc) or 'ValidationError' in type(exc).__name__:
            from langchain_core.messages import ToolMessage
            print(f'  [WARN] Malformed tool call (missing field) — scoring 0.0')
            return ToolMessage(content='0.0', tool_call_id=tool_call.get('id', 'unknown'))
        raise


def run_experiment(use_anonymized: bool, verbose: bool = True) -> dict:
    condition = 'anonymized' if use_anonymized else 'original'
    n_entries = normal_entries_anon if use_anonymized else normal_entries_orig
    a_entries = attack_entries_anon if use_anonymized else attack_entries_orig

    chain = prompt | llm_with_tool
    n, msgs, train_f1s = 0, [], []
    last_ai_msg = None

    while n < N_ROUNDS:
        ai_msg = invoke_with_retry(chain, {
            'k': K_RULES,
            'normal_entries': json.dumps(n_entries),
            'attack_entries': json.dumps(a_entries),
            'msgs': msgs,
        })
        last_ai_msg = ai_msg

        tool_msgs = [safe_invoke_tool(tc) for tc in ai_msg.tool_calls]
        mean_f1 = (
            sum(float(m.content) for m in tool_msgs) / len(tool_msgs)
            if tool_msgs else 0.0
        )
        train_f1s.append(mean_f1)

        feedback = HumanMessage(
            f'The current mean f1-score for the generated rules is {mean_f1:.4f}. '
            'If this mean f1-score is greater than the previous rounds, keep the better '
            'performing rules and revise or replace only the underperforming ones '
            '(those with a score less than mean). '
            'Otherwise, revise or replace any rules that have a score less than mean. '
            f'Based on the feedback, generate exactly {K_RULES} rules to filter attack '
            'entries and make a tool call for each rule, ensuring that a tool call is '
            'made for every entry every time.'
        )
        msgs.extend([ai_msg, *tool_msgs, feedback])
        n += 1

        token_usage = ai_msg.response_metadata.get('token_usage', {})
        if verbose:
            print(f'  [{condition}] Round {n}/{N_ROUNDS}  '
                  f'mean_f1={mean_f1:.4f}  '
                  f'tokens={token_usage.get("total_tokens", "?")}')
        if n < N_ROUNDS:
            time.sleep(INTER_ROUND_SLEEP)

    return {
        'condition':        condition,
        'train_f1s':        train_f1s,
        'final_tool_calls': last_ai_msg.tool_calls if last_ai_msg else [],
        'all_msgs':         msgs,
    }


print('run_experiment() ready.')


run_experiment() ready.


## Cell 7 — Multi-Seed Execution

In [11]:
################################################################################
# Cell 7 — Multi-Seed Execution
#
# Runs N_SEEDS independent repetitions of both conditions.
# Samples are fixed (computed in Cell 3); seed variation comes from LLM
# temperature=0.1 stochasticity.
################################################################################

import time

N_SEEDS          = 3
INTER_SEED_SLEEP = 60

print(f'=== Running ORIGINAL condition ({N_SEEDS} seeds) ===')
original_results = []
for seed in range(N_SEEDS):
    print(f'\n--- Original seed {seed+1}/{N_SEEDS} ---')
    result = run_experiment(use_anonymized=False, verbose=True)
    original_results.append(result)
    if seed < N_SEEDS - 1:
        print(f'  Sleeping {INTER_SEED_SLEEP}s...')
        time.sleep(INTER_SEED_SLEEP)

print(f'\n=== Running ANONYMIZED condition ({N_SEEDS} seeds) ===')
anon_results = []
for seed in range(N_SEEDS):
    print(f'\n--- Anonymized seed {seed+1}/{N_SEEDS} ---')
    result = run_experiment(use_anonymized=True, verbose=True)
    anon_results.append(result)
    if seed < N_SEEDS - 1:
        print(f'  Sleeping {INTER_SEED_SLEEP}s...')
        time.sleep(INTER_SEED_SLEEP)

print('\nAll runs complete.')


=== Running ORIGINAL condition (3 seeds) ===

--- Original seed 1/3 ---
  [original] Round 1/5  mean_f1=0.7067  tokens=4182
  [original] Round 2/5  mean_f1=0.6877  tokens=4639
  [original] Round 3/5  mean_f1=0.6989  tokens=5111
  [original] Round 4/5  mean_f1=0.7139  tokens=5584
  [original] Round 5/5  mean_f1=0.7142  tokens=6057
  Sleeping 60s...

--- Original seed 2/3 ---
  [original] Round 1/5  mean_f1=0.6606  tokens=4036
  [original] Round 2/5  mean_f1=0.6354  tokens=4558
  [original] Round 3/5  mean_f1=0.6780  tokens=5080
  [original] Round 4/5  mean_f1=0.6817  tokens=5575
  [original] Round 5/5  mean_f1=0.6832  tokens=6070
  Sleeping 60s...

--- Original seed 3/3 ---
  [original] Round 1/5  mean_f1=0.7080  tokens=4205
  [original] Round 2/5  mean_f1=0.7103  tokens=4725
  [original] Round 3/5  mean_f1=0.7073  tokens=5185
  [original] Round 4/5  mean_f1=0.7025  tokens=5645
  [original] Round 5/5  mean_f1=0.6913  tokens=6104

=== Running ANONYMIZED condition (3 seeds) ===

--- Anony

## Cell 8 — Test Set Evaluation

In [12]:
################################################################################
# Cell 8 — Test Set Evaluation
#
# Applies final tool calls from each seed to the held-out test set.
# Resolves anonymous feature names via reverse_map before lookup.
################################################################################

from statistics import mode
from sklearn.metrics import classification_report


def evaluate_rules_on_test(tool_calls: list) -> dict:
    if not tool_calls:
        print('  WARNING: empty tool_calls.')
        return {}
    datasets = {'normal': normal_df_test, 'attack': attack_df_test}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in range(len(dataset)):
            row = dataset.iloc[i]
            preds = []
            for tc in tool_calls:
                args      = tc['args']
                real_name = reverse_map.get(args['feature_name'], args['feature_name'])
                op, val   = args['op'], args['value']
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    val = str(val)
                if op in OPERATIONS:
                    try:
                        preds.append(
                            'attack' if OPERATIONS[op](row[real_name], val) else 'normal'
                        )
                    except TypeError:
                        preds.append('normal')
            y_true.append(label)
            y_pred.append(mode(preds) if preds else 'normal')
    return classification_report(
        y_true, y_pred, digits=4, output_dict=True, zero_division=0
    )


print('Evaluating original condition on test set...')
original_test_reports = [
    evaluate_rules_on_test(r['final_tool_calls']) for r in original_results
]
print('Evaluating anonymized condition on test set...')
anon_test_reports = [
    evaluate_rules_on_test(r['final_tool_calls']) for r in anon_results
]
print('Done.')


Evaluating original condition on test set...
Evaluating anonymized condition on test set...
Done.


## Cell 9 — Comparison Metrics

In [13]:
################################################################################
# Cell 9 — Comparison Metrics: F1, Jaccard, Semantic Categories
################################################################################

from collections import Counter
import numpy as np
from tabulate import tabulate

orig_f1s = [r['macro avg']['f1-score'] for r in original_test_reports]
anon_f1s = [r['macro avg']['f1-score'] for r in anon_test_reports]
delta_f1 = np.mean(orig_f1s) - np.mean(anon_f1s)


def extract_features(tool_calls: list) -> set:
    return {reverse_map.get(tc['args']['feature_name'], tc['args']['feature_name'])
            for tc in tool_calls}


orig_features_per_run = [extract_features(r['final_tool_calls']) for r in original_results]
anon_features_per_run = [extract_features(r['final_tool_calls']) for r in anon_results]
orig_all_features = Counter(f for fs in orig_features_per_run for f in fs)
anon_all_features = Counter(f for fs in anon_features_per_run for f in fs)


def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if (a | b) else 0.0


jaccards     = [jaccard(o, a) for o, a in zip(orig_features_per_run, anon_features_per_run)]
mean_jaccard = np.mean(jaccards)
std_jaccard  = np.std(jaccards)


def category_dist(feature_sets: list) -> Counter:
    counts = Counter()
    for fs in feature_sets:
        for f in fs:
            counts[get_category(f)] += 1
    return counts


orig_cat = category_dist(orig_features_per_run)
anon_cat = category_dist(anon_features_per_run)

print('\n=== Per-Seed Feature Selection ===')
seed_rows = []
for i in range(N_SEEDS):
    seed_rows.append([
        i + 1,
        ', '.join(sorted(orig_features_per_run[i])),
        f'{orig_f1s[i]:.4f}',
        ', '.join(sorted(anon_features_per_run[i])),
        f'{anon_f1s[i]:.4f}',
        f'{jaccards[i]:.3f}',
    ])
print(tabulate(seed_rows,
    headers=['Seed', 'Original features', 'Orig F1',
             'Anon features (real names)', 'Anon F1', 'Jaccard'],
    tablefmt='grid'))

print('\n=== Condition Summary ===')
print(tabulate([
    ['Original',   f'{np.mean(orig_f1s):.4f}', f'{np.std(orig_f1s):.4f}'],
    ['Anonymized', f'{np.mean(anon_f1s):.4f}', f'{np.std(anon_f1s):.4f}'],
], headers=['Condition', 'Mean Test F1', 'Std'], tablefmt='grid'))
print(f'\u0394F1 (original \u2212 anonymized): {delta_f1:+.4f}')
print(f'Mean Jaccard (feature overlap): {mean_jaccard:.3f} \u00b1 {std_jaccard:.3f}')

print('\n=== Semantic Category Distribution ===')
cat_rows = [
    [cat, orig_cat.get(cat, 0), anon_cat.get(cat, 0)]
    for cat in list(SEMANTIC_CATEGORIES.keys()) + ['unknown']
    if orig_cat.get(cat, 0) > 0 or anon_cat.get(cat, 0) > 0
]
print(tabulate(cat_rows,
    headers=['Category', 'Original (total)', 'Anonymized (total)'],
    tablefmt='grid'))

print('\n=== Most-Selected Features — Original ===')
print(tabulate([[f, c] for f, c in orig_all_features.most_common()],
    headers=['Feature', f'Freq (/{N_SEEDS} seeds)'], tablefmt='grid'))

print('\n=== Most-Selected Features — Anonymized ===')
print(tabulate([[f, c] for f, c in anon_all_features.most_common()],
    headers=['Feature (real name)', f'Freq (/{N_SEEDS} seeds)'], tablefmt='grid'))



=== Per-Seed Feature Selection ===
+--------+--------------------------------------+-----------+--------------------------------+-----------+-----------+
|   Seed | Original features                    |   Orig F1 | Anon features (real names)     |   Anon F1 |   Jaccard |
+========+======================================+===========+================================+===========+===========+
|      1 | ct_srv_src, dttl, dur, rate, sload   |    0.7698 | dload, dur, rate, sload, sloss |    0.769  |     0.429 |
+--------+--------------------------------------+-----------+--------------------------------+-----------+-----------+
|      2 | ct_srv_src, dttl, dur, rate, sload   |    0.7827 | dload, dur, rate, sload, sloss |    0.769  |     0.429 |
+--------+--------------------------------------+-----------+--------------------------------+-----------+-----------+
|      3 | ct_srv_src, dload, dttl, rate, sload |    0.7812 | dload, dttl, dur, rate, sload  |    0.7669 |     0.667 |
+--------+--

## Cell 10 — Save Results

In [14]:
################################################################################
# Cell 10 — Save Results to JSON
#
# Output: results/anon/unsw-nb15-anon-ablation.json
# Schema mirrors WUSTL-IIoT anon ablation for cross-dataset comparison.
################################################################################

import json, os
import numpy as np

os.makedirs('results/anon', exist_ok=True)


def report_to_serializable(report: dict) -> dict:
    return {
        k: {mk: float(mv) for mk, mv in v.items()} if isinstance(v, dict) else float(v)
        for k, v in report.items()
    }


payload = {
    'dataset':   dataset_name,
    'model':     model_name,
    'n_seeds':   N_SEEDS,
    'n_rounds':  N_ROUNDS,
    'k_rules':   K_RULES,
    'sample_size_per_class': SAMPLE_SIZE,
    'anon_map':  anon_map,
    'original': {
        'test_f1_mean':              float(np.mean(orig_f1s)),
        'test_f1_std':               float(np.std(orig_f1s)),
        'test_f1_per_seed':          [float(f) for f in orig_f1s],
        'train_f1s_per_seed':        [r['train_f1s'] for r in original_results],
        'selected_features_per_seed':[sorted(fs) for fs in orig_features_per_run],
        'feature_frequency':         dict(orig_all_features),
        'category_dist':             dict(orig_cat),
        'classification_reports':    [report_to_serializable(r)
                                      for r in original_test_reports],
    },
    'anonymized': {
        'test_f1_mean':              float(np.mean(anon_f1s)),
        'test_f1_std':               float(np.std(anon_f1s)),
        'test_f1_per_seed':          [float(f) for f in anon_f1s],
        'train_f1s_per_seed':        [r['train_f1s'] for r in anon_results],
        'selected_features_per_seed':[sorted(fs) for fs in anon_features_per_run],
        'feature_frequency':         dict(anon_all_features),
        'category_dist':             dict(anon_cat),
        'classification_reports':    [report_to_serializable(r)
                                      for r in anon_test_reports],
    },
    'delta_f1':          float(delta_f1),
    'mean_jaccard':      float(mean_jaccard),
    'std_jaccard':       float(std_jaccard),
    'jaccards_per_seed': [float(j) for j in jaccards],
}

out_path = f'results/anon/{dataset_name}-anon-ablation.json'
with open(out_path, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Results saved to: {out_path}')
print(f'\nKey findings:')
print(f'  Original   F1: {np.mean(orig_f1s):.4f} \u00b1 {np.std(orig_f1s):.4f}')
print(f'  Anonymized F1: {np.mean(anon_f1s):.4f} \u00b1 {np.std(anon_f1s):.4f}')
print(f'  \u0394F1:           {delta_f1:+.4f}')
print(f'  Mean Jaccard:  {mean_jaccard:.3f} \u00b1 {std_jaccard:.3f}')
print(f'\nValues for tex:')
print(f'  [INSERT: f1_orig_unsw] = {np.mean(orig_f1s):.4f}')
print(f'  [INSERT: f1_anon_unsw] = {np.mean(anon_f1s):.4f}')
print(f'  [INSERT: anon_f1_unsw] = {np.mean(anon_f1s):.4f}  (same as f1_anon_unsw)')


Results saved to: results/anon/unsw-nb15-anon-ablation.json

Key findings:
  Original   F1: 0.7779 ± 0.0058
  Anonymized F1: 0.7683 ± 0.0010
  ΔF1:           +0.0096
  Mean Jaccard:  0.508 ± 0.112

Values for tex:
  [INSERT: f1_orig_unsw] = 0.7779
  [INSERT: f1_anon_unsw] = 0.7683
  [INSERT: anon_f1_unsw] = 0.7683  (same as f1_anon_unsw)
